# Solution 1b — Confound check + confidence intervals

**Two jobs:**

**(1) The authorship confound.** Your 300 pilot MSA queries were written while reading their gold passages, so they reuse passage wording (0.561 token overlap vs 0.360 for an independent rewrite). That inflates the MSA baseline and therefore the measured dialect gap. The 200 Wikipedia items share no authorship with their passages — running both subsets shows how much of the gap survives once the confound is removed.

**(2) Statistics.** A 6-point gap on 300 items is ~18 questions. Without confidence intervals there's no way to know if it's real. This bootstraps every headline number, and uses *paired* resampling for differences since conditions share the same questions.

Runs DarijaBERT (α=0.8), multilingual-e5 (α=0.8) and MiniLM (α=0.2, control).

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **No API key.** Use a GPU runtime.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers requests

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "repo_raw": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main",
    # Best configuration from solutions_1_2. MiniLM is kept as a control so the
    # encoder effect can be reported on both subsets, not just the best model.
    "encoders": [
        ("SI2M-Lab/DarijaBERT", "mean", 0.8),
        ("intfloat/multilingual-e5-base", "st_e5", 0.8),
        ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "st", 0.2),
    ],
    "k_values": (1, 3, 5, 10),
    "max_k": 10,
    "bootstrap_n": 1000,   # resamples for confidence intervals
    "ci": 95,
}

### Load corpus and BOTH question sets

In [ ]:
import json, requests

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)
pilot_qa = json.loads(requests.get(f"{CONFIG['repo_raw']}/data/qa_pairs.json").text)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
known = set(corpus_ids)

SUBSETS = {
    "pilot": [q for q in pilot_qa if q["source_chunk_id"] in known],
    "wikipedia": [q for q in wiki_qa if q["source_chunk_id"] in known],
}

print(f"Corpus: {len(corpus)} passages")
for name, items in SUBSETS.items():
    print(f"  {name:<10} {len(items)} questions")

### Arabic normalization + BM25

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
print("BM25 index built.")

### Rule-based mitigation (same lexicon as solutions_1_2)

In [ ]:
DARIJA_TO_MSA = {
    "شنو هي": "ما هي", "شنو هو": "ما هو", "شنو هما": "ما هي", "شنو هوما": "ما هي",
    "شحال ديال": "كم", "بشحال": "بكم", "شحال": "كم",
    "فين": "أين", "لفين": "إلى أين", "منين": "من أين",
    "علاش": "لماذا", "علاه": "لماذا", "كيفاش": "كيف",
    "إمتى": "متى", "فوقاش": "متى", "شكون": "من", "أشمن": "أي",
    "فأش": "في أي", "فأي": "في أي", "بأش": "بماذا", "أش": "ماذا", "شنو": "ما",
    "واش": "هل", "كاينة": "توجد", "كاين": "يوجد", "ماكاينش": "لا يوجد",
    "بزاف": "كثيرا", "دابا": "الآن", "غادي": "سوف", "باش": "لكي",
    "هادشي": "هذا", "هادي": "هذه", "هاد": "هذا",
    "بحال": "مثل", "حيت": "لأن", "ملي": "عندما", "واخا": "رغم",
    "ماشي": "ليس", "بلا": "بدون", "ديالو": "", "ديالها": "", "ديالهم": "", "ديال": "",
}

def rule_normalize(text):
    out = text
    for d, m in sorted(DARIJA_TO_MSA.items(), key=lambda kv: -len(kv[0])):
        out = re.sub(rf"(?<!\w){re.escape(d)}(?!\w)", m, out)
    return re.sub(r"\s+", " ", out).strip()

for items in SUBSETS.values():
    for it in items:
        it["M4_rulebased"] = rule_normalize(it["darija_query"])
        it["M3_rule_expansion"] = f'{it["darija_query"]} {it["M4_rulebased"]}'

VARIANTS = ["msa_query", "darija_query", "M4_rulebased", "M3_rule_expansion"]
print("Mitigation variants built for both subsets.")

### Encoder loading

In [ ]:
import torch, gc
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

def encode_corpus(model_id, kind):
    if kind.startswith("st"):
        model = SentenceTransformer(model_id)
        pp, pq = ("passage: ", "query: ") if kind == "st_e5" else ("", "")
        emb = model.encode([pp + t for t in corpus_texts],
                           normalize_embeddings=True, show_progress_bar=True, batch_size=64)
        return (lambda q: model.encode([pq + q], normalize_embeddings=True)[0]), np.asarray(emb, "float32")

    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModel.from_pretrained(model_id).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    mdl.to(dev)

    def embed(texts, bs=32):
        out = []
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i + bs], padding=True, truncation=True,
                      max_length=256, return_tensors="pt").to(dev)
            with torch.no_grad():
                h = mdl(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).float()
            p = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            out.append(torch.nn.functional.normalize(p, p=2, dim=1).cpu().numpy())
        return np.vstack(out)

    return (lambda q: embed([q])[0]), np.asarray(embed(corpus_texts), "float32")

### Retrieval; record per-item hits so bootstrapping is possible

In [ ]:
def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def make_retriever(encode_query, corpus_emb):
    def retrieve(q, k, alpha):
        s = 0.0
        if alpha > 0:
            s = alpha * minmax(corpus_emb @ encode_query(q))
        if alpha < 1:
            s = s + (1 - alpha) * minmax(np.asarray(bm25.get_scores(tokenize(q))))
        return [corpus_ids[i] for i in np.argsort(-s)[:k]]
    return retrieve

def per_item(retrieve, items, field, alpha):
    """Returns per-item hit vectors and reciprocal ranks, which the bootstrap
    resamples. Aggregating first would make CIs impossible."""
    hits = {k: [] for k in CONFIG["k_values"]}
    rr = []
    for it in items:
        got = retrieve(it[field], CONFIG["max_k"], alpha)
        gold = it["source_chunk_id"]
        for k in CONFIG["k_values"]:
            hits[k].append(1.0 if gold in got[:k] else 0.0)
        rr.append(1.0 / (got.index(gold) + 1) if gold in got else 0.0)
    return {**{f"R@{k}": np.array(v) for k, v in hits.items()}, "MRR": np.array(rr)}

### Run both subsets across all encoders

In [ ]:
raw = {}   # (encoder, subset, variant) -> per-item arrays

for model_id, kind, alpha in CONFIG["encoders"]:
    short = model_id.split("/")[-1]
    print(f"\n=== {short} (alpha={alpha}) ===")
    try:
        encode_query, corpus_emb = encode_corpus(model_id, kind)
    except Exception as e:
        print(f"  SKIPPED ({type(e).__name__}: {e})")
        continue
    retrieve = make_retriever(encode_query, corpus_emb)

    for subset, items in SUBSETS.items():
        for v in VARIANTS:
            raw[(short, subset, v)] = per_item(retrieve, items, v, alpha)
        print(f"  {subset} done ({len(items)} questions)")

    del encode_query, corpus_emb, retrieve
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nCollected {len(raw)} result vectors.")

### Bootstrap confidence intervals

In [ ]:
rng = np.random.default_rng(42)

def boot_ci(vec, n=None, ci=None):
    n = n or CONFIG["bootstrap_n"]
    ci = ci or CONFIG["ci"]
    idx = rng.integers(0, len(vec), size=(n, len(vec)))
    means = vec[idx].mean(axis=1)
    lo, hi = np.percentile(means, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return vec.mean(), lo, hi

def boot_diff_ci(a, b, n=None, ci=None):
    """CI on the paired difference a - b. Paired because both conditions are
    evaluated on the same questions; treating them as independent would
    overstate the uncertainty."""
    n = n or CONFIG["bootstrap_n"]
    ci = ci or CONFIG["ci"]
    d = a - b
    idx = rng.integers(0, len(d), size=(n, len(d)))
    means = d[idx].mean(axis=1)
    lo, hi = np.percentile(means, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return d.mean(), lo, hi

print(f"Bootstrap: {CONFIG['bootstrap_n']} resamples, {CONFIG['ci']}% CI\n")

### Main table: dialect gap per subset, with CIs

In [ ]:
import pandas as pd

rows = []
for (enc, subset, variant), metrics in raw.items():
    for metric, vec in metrics.items():
        m, lo, hi = boot_ci(vec)
        rows.append({"encoder": enc, "subset": subset, "variant": variant,
                     "metric": metric, "value": m, "lo": lo, "hi": hi})

results = pd.DataFrame(rows)
results.to_csv("results_with_ci.csv", index=False)

METRIC = "R@5"
print(f"=== {METRIC} by subset (95% CI) ===\n")
for enc in results.encoder.unique():
    print(f"--- {enc} ---")
    for subset in SUBSETS:
        sub = results[(results.encoder == enc) & (results.subset == subset) &
                      (results.metric == METRIC)].set_index("variant")
        if sub.empty:
            continue
        line = []
        for v in VARIANTS:
            r = sub.loc[v]
            line.append(f"{v:<18} {r['value']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}]")
        print(f"  {subset}:")
        for l in line:
            print(f"    {l}")
    print()

### The confound test: is the dialect gap smaller on Wikipedia items?

In [ ]:
print("=== DIALECT GAP (MSA - Darija), paired bootstrap 95% CI ===\n")
gap_rows = []
for enc in {e for (e, _, _) in raw}:
    for subset in SUBSETS:
        key_m = (enc, subset, "msa_query")
        key_d = (enc, subset, "darija_query")
        if key_m not in raw:
            continue
        for metric in ["R@1", "R@5", "MRR"]:
            d, lo, hi = boot_diff_ci(raw[key_m][metric], raw[key_d][metric])
            gap_rows.append({"encoder": enc, "subset": subset, "metric": metric,
                             "gap": d, "lo": lo, "hi": hi,
                             "significant": "yes" if lo > 0 else "no"})

gaps = pd.DataFrame(gap_rows).sort_values(["encoder", "metric", "subset"])
print(gaps.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
gaps.to_csv("dialect_gaps.csv", index=False)

print("\nIf the Wikipedia gap is clearly smaller than the pilot gap, part of the")
print("pilot gap came from question/passage authorship overlap, not from dialect.")

### Mitigation recovery per subset, with CIs on the improvement

In [ ]:
print("\n=== MITIGATION: improvement over raw Darija (paired 95% CI) ===\n")
mit_rows = []
for enc in {e for (e, _, _) in raw}:
    for subset in SUBSETS:
        kd = (enc, subset, "darija_query")
        if kd not in raw:
            continue
        base = raw[(enc, subset, "msa_query")]["R@5"].mean()
        mism = raw[kd]["R@5"].mean()
        drop = base - mism
        for v in ["M4_rulebased", "M3_rule_expansion"]:
            d, lo, hi = boot_diff_ci(raw[(enc, subset, v)]["R@5"], raw[kd]["R@5"])
            mit_rows.append({
                "encoder": enc, "subset": subset, "mitigation": v,
                "improvement": d, "lo": lo, "hi": hi,
                "recovery_%": (d / drop * 100) if drop > 0 else float("nan"),
                "significant": "yes" if lo > 0 else "no",
            })

mits = pd.DataFrame(mit_rows).sort_values("recovery_%", ascending=False)
print(mits.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
mits.to_csv("mitigation_ci.csv", index=False)

print("\n'significant = yes' means the 95% CI on the improvement excludes zero.")
print("A positive recovery_% with significant = no is not yet a real effect.")

### Download

In [ ]:
from google.colab import files
files.download("results_with_ci.csv")
files.download("dialect_gaps.csv")
files.download("mitigation_ci.csv")